# compare_extraction_methods notebook

This notebook embeds the full code from `compare_extraction_methods.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

"""
Compare areal (catchment-weighted) vs nearest-grid-cell extraction for six example basins.

Outputs under Results/extraction_comparison/:
  - forcing_comparison.csv       (bias-corrected forcing stats)
  - factors_pr_comparison.csv    (DOY correction factors for precipitation)
  - deficits_hydro_comparison.csv (annual ΣD from future Q vs Q80)
  - summary_for_thesis.csv       (one row per basin, key % differences)
  - plots/deficit_hydro_{basin}.png (areal vs nearest annual ΣD)

Run after both pipelines exist:
  python run_future_pipeline_6basins.py --extraction areal
  python run_future_pipeline_6basins.py --extraction nearest --workers 2

  python compare_extraction_methods.py
"""

from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from calibration_io import BASIN_IDS, basin_hydrograph_title
from compute_q80_deficits import (
    aggregate_hydro,
    aggregate_season_year,
    deficit_block,
    hydrological_year,
    load_threshold_series,
)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def dirs_for_method(method: str, root: Path) -> dict[str, Path]:
    suffix = "" if method == "areal" else "_nearest"
    return {
        "bc": root / f"Results/eurocordex_bias_corrected{suffix}",
        "forcing": root / f"Results/forcing_future{suffix}",
        "discharge": root / f"Results/discharge_components_future{suffix}",
        "deficit": root / f"Results/deficit_q80{suffix}",
    }


def load_bc_series(bc_dir: Path, basin_id: str, var: str, col: str) -> pd.Series:
    path = bc_dir / f"{var}_bc_daily_{basin_id}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    return pd.Series(pd.to_numeric(df[col], errors="coerce").values, index=df["date"])


def forcing_stats(bc_dir: Path, basin_id: str, ref_start: str, ref_end: str) -> dict:
    pr = load_bc_series(bc_dir, basin_id, "pr", "pr_bc")
    tas = load_bc_series(bc_dir, basin_id, "tas", "tas_bc")
    pet_path = bc_dir / f"pet_future_bc_{basin_id}.csv"
    pet = pd.read_csv(pet_path)
    pet["date"] = pd.to_datetime(pet["date"]).dt.normalize()
    pet_s = pd.Series(pet["pet_mm_day"].values, index=pet["date"])

    ref = slice(ref_start, ref_end)
    fut = slice("2006-01-01", "2100-12-31")
    return {
        "pr_mean_ref": float(pr.loc[ref].mean()),
        "pr_mean_fut": float(pr.loc[fut].mean()),
        "tas_mean_ref": float(tas.loc[ref].mean()),
        "tas_mean_fut": float(tas.loc[fut].mean()),
        "pet_mean_fut": float(pet_s.loc[fut].mean()),
    }


def compare_factors(bc_areal: Path, bc_near: Path, basin_id: str) -> dict:
    fa = pd.read_csv(bc_areal / f"factors_pr_{basin_id}.csv", index_col="doy")["factor"]
    fn = pd.read_csv(bc_near / f"factors_pr_{basin_id}.csv", index_col="doy")["factor"]
    common = fa.index.intersection(fn.index)
    a, n = fa.loc[common].values, fn.loc[common].values
    diff = n - a
    return {
        "factor_pr_rmse": float(np.sqrt(np.mean(diff**2))),
        "factor_pr_mean_abs_diff": float(np.mean(np.abs(diff))),
        "factor_pr_max_abs_diff": float(np.max(np.abs(diff))),
        "factor_pr_corr": float(np.corrcoef(a, n)[0, 1]) if len(common) > 2 else np.nan,
    }


def compute_deficits_for_q(
    q_csv: Path,
    q80_csv: Path,
    basin_ids: list[str],
    hydro_start_month: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    q_df = pd.read_csv(q_csv)
    q_df["date"] = pd.to_datetime(q_df["date"]).dt.normalize()
    q80_wide = pd.read_csv(q80_csv)
    dates = pd.DatetimeIndex(q_df["date"].values)
    parts = []
    for bid in basin_ids:
        if bid not in q_df.columns:
            continue
        q80 = load_threshold_series(dates, q80_wide, bid)
        q_sim = pd.to_numeric(q_df[bid], errors="coerce").to_numpy(dtype=float)
        parts.append(deficit_block(dates, q_sim, q80, bid))
    daily = pd.concat(parts, ignore_index=True)
    return aggregate_hydro(daily, hydro_start_month), aggregate_season_year(daily)


def deficit_summary_metrics(hydro: pd.DataFrame, basin_id: str) -> dict:
    sub = hydro.loc[hydro["basin_id"] == basin_id].copy()
    sub = sub.loc[sub["hydro_year"] >= 2006]
    if sub.empty:
        return {}
    late = sub.loc[sub["hydro_year"] >= 2071]
    return {
        "mean_annual_sum_D": float(sub["sum_D_mm"].mean()),
        "max_hydro_year_sum_D": float(sub["sum_D_mm"].max()),
        "max_hydro_year": int(sub.loc[sub["sum_D_mm"].idxmax(), "hydro_year"]),
        "mean_sum_D_2071_2100": float(late["sum_D_mm"].mean()) if len(late) else np.nan,
        "total_sum_D_2006_2100": float(sub["sum_D_mm"].sum()),
    }


def plot_deficit_comparison(
    hydro_areal: pd.DataFrame,
    hydro_near: pd.DataFrame,
    basin_id: str,
    out_png: Path,
) -> None:
    a = hydro_areal.loc[hydro_areal["basin_id"] == basin_id].sort_values("hydro_year")
    n = hydro_near.loc[hydro_near["basin_id"] == basin_id].sort_values("hydro_year")
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(a["hydro_year"], a["sum_D_mm"], color="#1f538d", linewidth=1.2, label="Areal (weighted)")
    ax.plot(n["hydro_year"], n["sum_D_mm"], color="#c0392b", linewidth=1.0, alpha=0.85, label="Nearest cell")
    ax.set_title(f"{basin_hydrograph_title(basin_id)}\nHydrological-year ΣD (mm)")
    ax.set_xlabel("Hydrological year")
    ax.set_ylabel("ΣD (mm)")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=150)
    plt.close(fig)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawTextHelpFormatter)
    p.add_argument("--root", type=Path, default=ROOT)
    p.add_argument("--ref_start", default="1976-01-01")
    p.add_argument("--ref_end", default="2005-12-31")
    p.add_argument("--hydro_start_month", type=int, default=10)
    p.add_argument(
        "--q80_csv",
        type=Path,
        default=None,
        help="Q80 thresholds (default: Results/thresholds/Q80_daily_thresholds_last30y.csv).",
    )
    p.add_argument(
        "--basins",
        default=",".join(BASIN_IDS),
        help="Comma-separated basin IDs.",
    )
    return p.parse_known_args()[0]


def main() -> None:
    args = parse_args()
    root = args.root.resolve()
    basin_ids = [b.strip() for b in args.basins.split(",") if b.strip()]
    q80_csv = args.q80_csv or (root / "Results" / "thresholds" / "Q80_daily_thresholds_last30y.csv")
    out_dir = root / "Results" / "extraction_comparison"
    plot_dir = out_dir / "plots"
    out_dir.mkdir(parents=True, exist_ok=True)

    d_areal = dirs_for_method("areal", root)
    d_near = dirs_for_method("nearest", root)

    if not d_areal["bc"].is_dir():
        raise FileNotFoundError(f"Missing areal BC dir: {d_areal['bc']}")
    if not d_near["bc"].is_dir():
        raise FileNotFoundError(
            f"Missing nearest BC dir: {d_near['bc']}\n"
            "Run: python run_future_pipeline_6basins.py --extraction nearest --workers 2"
        )

    forcing_rows: list[dict] = []
    factor_rows: list[dict] = []
    summary_rows: list[dict] = []

    for bid in basin_ids:
        wpath = d_areal["bc"] / f"grid_weights_{bid}.csv"
        n_cells = len(pd.read_csv(wpath)) if wpath.exists() else np.nan

        sa = forcing_stats(d_areal["bc"], bid, args.ref_start, args.ref_end)
        sn = forcing_stats(d_near["bc"], bid, args.ref_start, args.ref_end)
        forcing_rows.append({"basin_id": bid, "method": "areal", "n_grid_cells": n_cells, **sa})
        forcing_rows.append({"basin_id": bid, "method": "nearest", "n_grid_cells": 1, **sn})

        fc: dict = {}
        if (d_areal["bc"] / f"factors_pr_{bid}.csv").exists() and (d_near["bc"] / f"factors_pr_{bid}.csv").exists():
            fc = compare_factors(d_areal["bc"], d_near["bc"], bid)
            factor_rows.append({"basin_id": bid, **fc})

        row = {"basin_id": bid, "n_grid_cells_areal": n_cells}
        row["pr_bc_mean_ref_pct_diff"] = 100 * (sn["pr_mean_ref"] - sa["pr_mean_ref"]) / max(sa["pr_mean_ref"], 1e-9)
        row["pr_bc_mean_fut_pct_diff"] = 100 * (sn["pr_mean_fut"] - sa["pr_mean_fut"]) / max(sa["pr_mean_fut"], 1e-9)
        row.update(fc)
        summary_rows.append(row)

    pd.DataFrame(forcing_rows).to_csv(out_dir / "forcing_comparison.csv", index=False)
    if factor_rows:
        pd.DataFrame(factor_rows).to_csv(out_dir / "factors_pr_comparison.csv", index=False)

    if not d_areal["discharge"].joinpath("Q_total_all_basins.csv").exists():
        print(f"Missing {d_areal['discharge'] / 'Q_total_all_basins.csv'} — run areal pipeline first.")
    elif not d_near["discharge"].joinpath("Q_total_all_basins.csv").exists():
        print(f"Missing nearest Q CSV — run nearest pipeline through DPWM step.")
    else:
        hydro_a, season_a = compute_deficits_for_q(
            d_areal["discharge"] / "Q_total_all_basins.csv", q80_csv, basin_ids, args.hydro_start_month,
        )
        hydro_n, season_n = compute_deficits_for_q(
            d_near["discharge"] / "Q_total_all_basins.csv", q80_csv, basin_ids, args.hydro_start_month,
        )
        hydro_a["method"] = "areal"
        hydro_n["method"] = "nearest"
        pd.concat([hydro_a, hydro_n], ignore_index=True).to_csv(
            out_dir / "deficits_hydro_year_by_method.csv", index=False,
        )
        season_a["method"] = "areal"
        season_n["method"] = "nearest"
        pd.concat([season_a, season_n], ignore_index=True).to_csv(
            out_dir / "deficits_season_year_by_method.csv", index=False,
        )

        deficit_cmp_rows = []
        for bid in basin_ids:
            ma = deficit_summary_metrics(hydro_a, bid)
            mn = deficit_summary_metrics(hydro_n, bid)
            plot_deficit_comparison(hydro_a, hydro_n, bid, plot_dir / f"deficit_hydro_{bid}.png")
            for key in ma:
                pct = 100 * (mn[key] - ma[key]) / max(abs(ma[key]), 1e-9)
                deficit_cmp_rows.append({
                    "basin_id": bid,
                    "metric": key,
                    "areal": ma[key],
                    "nearest": mn[key],
                    "pct_diff_nearest_vs_areal": pct,
                })
                for i, sr in enumerate(summary_rows):
                    if sr["basin_id"] == bid:
                        summary_rows[i][key] = ma[key]
                        summary_rows[i][f"{key}_nearest"] = mn[key]
                        summary_rows[i][f"{key}_pct_diff"] = pct
        pd.DataFrame(deficit_cmp_rows).to_csv(out_dir / "deficits_metrics_comparison.csv", index=False)

    pd.DataFrame(summary_rows).to_csv(out_dir / "summary_for_thesis.csv", index=False)
    print(f"Saved comparison tables and plots under: {out_dir}")


# Execute the script entry point
main()
